In [0]:
%sql
SELECT * FROM read_files(
 '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
 format => 'csv',
 header => true
 )


In [0]:
%sql
-- Crear el schema bronze si no existe
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze;
-- Primero, eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;
-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
CREATE TABLE bootcamp.bronze.properties_bronze
 SELECT * FROM read_files(
 '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
 format => 'csv',
 header => true
 )
 where url like 'https%' --Filtramos por aquellos links que sean consistentes
 ;

# Exploración Incicial y Análisis de Calidad

## Exploración Inicial

In [0]:
%sql
Select Count(*)
FROM bootcamp.bronze.properties_bronze

In [0]:
%sql
Describe bootcamp.bronze.properties_bronze

In [0]:
%sql
SELECT id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales, antiguedad, estado, zona
FROM bootcamp.bronze.properties_bronze
LIMIT 10

## Analisis de valores nulos

In [0]:
%sql
WITH conteos AS (
    SELECT
        COUNT(*) AS total_filas,
        COUNT(precio)                     AS nn_precio,
        COUNT(expensas)                  AS nn_expensas,
        COUNT(tipo_de_operacion)         AS nn_tipo_de_operacion,
        COUNT(moneda)                    AS nn_moneda,
        COUNT(ambientes)                 AS nn_ambientes,
        COUNT(metros_cuadrados_totales)  AS nn_m2_totales,
        COUNT(metros_cuadrados_cubiertos) AS nn_m2_cubiertos,
        COUNT(orientacion_cardinal)      AS nn_orientacion_cardinal,
        COUNT(piso)                      AS nn_piso,
        COUNT(cochera)                   AS nn_cochera,
        COUNT(antiguedad)                AS nn_antiguedad,
        COUNT(estado)                    AS nn_estado,
        COUNT(zona)                      AS nn_zona
    FROM bootcamp.bronze.properties_bronze
)
SELECT
    total_filas,
    total_filas - nn_precio                    AS nulos_precio,
    total_filas - nn_expensas                  AS nulos_expensas,
    total_filas - nn_tipo_de_operacion         AS nulos_tipo_de_operacion,
    total_filas - nn_moneda                    AS nulos_moneda,
    total_filas - nn_ambientes                 AS nulos_ambientes,
    total_filas - nn_m2_totales                AS nulos_m2_totales,
    total_filas - nn_m2_cubiertos              AS nulos_m2_cubiertos,
    total_filas - nn_orientacion_cardinal      AS nulos_orientacion_cardinal,
    total_filas - nn_piso                      AS nulos_piso,
    total_filas - nn_cochera                   AS nulos_cochera,
    total_filas - nn_antiguedad                AS nulos_antiguedad,
    total_filas - nn_estado                    AS nulos_estado,
    total_filas - nn_zona                      AS nulos_zona
FROM conteos;

In [0]:
%sql
SELECT
    ROUND((COUNT(*) - COUNT(precio)) * 100.0 / COUNT(*), 2)                    AS pct_nulos_precio,
    ROUND((COUNT(*) - COUNT(expensas)) * 100.0 / COUNT(*), 2)                  AS pct_nulos_expensas,
    ROUND((COUNT(*) - COUNT(ambientes)) * 100.0 / COUNT(*), 2)                 AS pct_nulos_ambientes,
    ROUND((COUNT(*) - COUNT(metros_cuadrados_totales)) * 100.0 / COUNT(*), 2)  AS pct_nulos_m2_totales,
    ROUND((COUNT(*) - COUNT(metros_cuadrados_cubiertos)) * 100.0 / COUNT(*), 2) AS pct_nulos_m2_cubiertos,
    ROUND((COUNT(*) - COUNT(orientacion_cardinal)) * 100.0 / COUNT(*), 2)      AS pct_nulos_orientacion_cardinal,
    ROUND((COUNT(*) - COUNT(antiguedad)) * 100.0 / COUNT(*), 2)                AS pct_nulos_antiguedad
FROM bootcamp.bronze.properties_bronze;

In [0]:
%sql
with nulos as (
    SELECT
    ROUND((COUNT(*) - COUNT(precio)) * 100.0 / COUNT(*), 2)                    AS pct_nulos_precio,
    ROUND((COUNT(*) - COUNT(expensas)) * 100.0 / COUNT(*), 2)                  AS pct_nulos_expensas,
    ROUND((COUNT(*) - COUNT(ambientes)) * 100.0 / COUNT(*), 2)                 AS pct_nulos_ambientes,
    ROUND((COUNT(*) - COUNT(metros_cuadrados_totales)) * 100.0 / COUNT(*), 2)  AS pct_nulos_m2_totales,
    ROUND((COUNT(*) - COUNT(metros_cuadrados_cubiertos)) * 100.0 / COUNT(*), 2) AS pct_nulos_m2_cubiertos,
    ROUND((COUNT(*) - COUNT(orientacion_cardinal)) * 100.0 / COUNT(*), 2)      AS pct_nulos_orientacion_cardinal,
    ROUND((COUNT(*) - COUNT(antiguedad)) * 100.0 / COUNT(*), 2)                AS pct_nulos_antiguedad
FROM bootcamp.bronze.properties_bronze) 

SELECT * 
FROM nulos
WHERE 
    pct_nulos_precio > 50 OR
    pct_nulos_expensas > 50 OR
    pct_nulos_ambientes > 50 OR
    pct_nulos_m2_totales > 50 OR
    pct_nulos_m2_cubiertos > 50 OR
    pct_nulos_orientacion_cardinal > 50 OR
    pct_nulos_antiguedad > 50;



## Cardinalidad y Distribución

In [0]:
%sql

with total as (
    SELECT COUNT(*) AS total
    FROM bootcamp.bronze.properties_bronze
)

SELECT 
    tipo_de_operacion AS tipos, 
    COUNT(*) AS cantidad,
    ROUND(
        COUNT(*) * 100.0 / SUM(total.total), 
        2
    ) AS porcentaje
FROM bootcamp.bronze.properties_bronze
CROSS JOIN total
GROUP BY tipo_de_operacion, total.total
ORDER BY cantidad DESC;

In [0]:
%sql
SELECT moneda, COUNT(*) AS total
FROM bootcamp.bronze.properties_bronze
GROUP BY moneda



In [0]:
%sql
SELECT ambientes AS cantidad_ambientes, COUNT(*) AS propiedades
FROM bootcamp.bronze.properties_bronze
GROUP BY ambientes
ORDER BY ambientes ASC;

In [0]:
%sql
Select zona, Count(*) as propiedades
FROM bootcamp.bronze.properties_bronze
GROUP BY zona
Order BY propiedades DESC
Limit 15

In [0]:
%sql
SELECT estado, COUNT(*) AS cantidad
FROM bootcamp.bronze.properties_bronze
GROUP BY estado
ORDER BY cantidad DESC;


## Estadisticas Descriptivas

In [0]:
%sql

SELECT COUNT(*) as propiedades, moneda, MIN(CAST(precio AS DOUBLE)) as minimo, MAX(CAST(precio AS DOUBLE)) as maximo, AVG(CAST(precio AS DOUBLE)) as promedio, PERCENTILE(CAST(precio AS DOUBLE), 0.25) AS percentil_25, PERCENTILE(CAST(precio AS DOUBLE), 0.75) AS percentil_75
FROM bootcamp.bronze.properties_bronze
WHERE CAST(precio AS DOUBLE) > 0
GROUP BY moneda

In [0]:
%sql
SELECT COUNT(*) as propiedades, tipo_de_operacion, MIN(CAST(precio AS DOUBLE)) as minimo, MAX(CAST(precio AS DOUBLE)) as maximo, AVG(CAST(precio AS DOUBLE)) as promedio, PERCENTILE(CAST(precio AS DOUBLE), 0.25) AS percentil_25, PERCENTILE(CAST(precio AS DOUBLE), 0.75) AS percentil_75
FROM bootcamp.bronze.properties_bronze
WHERE CAST(precio AS DOUBLE) > 0
Group BY tipo_de_operacion

In [0]:
%sql
WITH metros_cuadrados_totales AS (
    SELECT 
        MIN(CAST(metros_cuadrados_totales AS DOUBLE)) AS minimo, 
        MAX(CAST(metros_cuadrados_totales AS DOUBLE)) AS maximo, 
        AVG(CAST(metros_cuadrados_totales AS DOUBLE)) AS promedio,
        PERCENTILE(CAST(metros_cuadrados_totales AS DOUBLE), 0.5) AS mediana
    FROM bootcamp.bronze.properties_bronze
    WHERE CAST(metros_cuadrados_totales AS DOUBLE) > 0
), 
metros_cuadrados_cubiertos AS (
    SELECT 
        MIN(CAST(metros_cuadrados_cubiertos AS DOUBLE)) AS minimo, 
        MAX(CAST(metros_cuadrados_cubiertos AS DOUBLE)) AS maximo, 
        AVG(CAST(metros_cuadrados_cubiertos AS DOUBLE)) AS promedio,
        PERCENTILE(CAST(metros_cuadrados_cubiertos AS DOUBLE), 0.5) AS mediana
    FROM bootcamp.bronze.properties_bronze
    WHERE CAST(metros_cuadrados_cubiertos AS DOUBLE) > 0
)
SELECT 'metros_cuadrados_totales' AS metrica, minimo, maximo, promedio, mediana
FROM metros_cuadrados_totales
UNION ALL
SELECT 'metros_cuadrados_cubiertos' AS metrica, minimo, maximo, promedio, mediana
FROM metros_cuadrados_cubiertos

In [0]:
%sql
SELECT antiguedad, COUNT(*) AS cantidad
FROM bootcamp.bronze.properties_bronze
GROUP BY antiguedad
ORDER BY cantidad DESC
LIMIT 20

In [0]:
%sql
SELECT DISTINCT antiguedad,
    COUNT(*) OVER(PARTITION BY antiguedad) AS registros,
    MAX(antiguedad) OVER() AS maximo,
    MIN(antiguedad) OVER() AS minimo,
    AVG(try_cast(antiguedad AS DOUBLE)) OVER() AS promedio
FROM bootcamp.bronze.properties_bronze
WHERE antiguedad != 999.0 AND antiguedad IS NOT NULL AND try_cast(antiguedad AS DOUBLE) >= 0

## Detección de Problemas de Calidas

In [0]:
%sql
WITH total_registros AS (
    SELECT COUNT(*) AS total
    FROM bootcamp.bronze.properties_bronze
), metros_cuadrados_invalidos AS (
    SELECT COUNT(*) AS metros_cuadrados_invalidos
    FROM bootcamp.bronze.properties_bronze
    WHERE metros_cuadrados_totales IS NULL OR TRY_CAST(metros_cuadrados_totales AS DOUBLE) <= 0
), precios_invalidos AS(
    SELECT COUNT(*) AS precios_invalidos
    FROM bootcamp.bronze.properties_bronze
    WHERE precio IS NULL OR TRY_CAST(precio AS DOUBLE) <= 0
), Moneda_vacia AS (
    SELECT COUNT(*) AS Moneda_vacia
    FROM bootcamp.bronze.properties_bronze
    WHERE moneda IS NULL OR moneda = ''
), operacion_vacia AS (
    SELECT COUNT(*) AS operacion_vacia
    FROM bootcamp.bronze.properties_bronze
    WHERE tipo_de_operacion IS NULL OR tipo_de_operacion = ''
), antiguedad_999 AS (
    SELECT COUNT(*) AS antiguedad_999
    FROM bootcamp.bronze.properties_bronze
    WHERE TRY_CAST(antiguedad AS DOUBLE) = 999
), ambientes_invalidos AS (
    SELECT COUNT(*) AS ambientes_invalidos
    FROM bootcamp.bronze.properties_bronze
    WHERE ambientes IS NULL 
       OR TRY_CAST(ambientes AS DOUBLE) = 0
), tipo_operacion_vacia AS (
    SELECT COUNT(*) AS tipo_operacion_vacia
    FROM bootcamp.bronze.properties_bronze
    WHERE tipo_de_operacion IS NULL 
       OR TRIM(tipo_de_operacion) = ''
)

SELECT 
    t.total,

    p.precios_invalidos,
    ROUND(p.precios_invalidos * 100.0 / t.total, 2) AS porcentaje_precios_invalidos,

    m.metros_cuadrados_invalidos,
    ROUND(m.metros_cuadrados_invalidos * 100.0 / t.total, 2) AS porcentaje_metros_cuadrados_invalidos,

    a.antiguedad_999,
    ROUND(a.antiguedad_999 * 100.0 / t.total, 2) AS porcentaje_antiguedad_999,

    amb.ambientes_invalidos,
    ROUND(amb.ambientes_invalidos * 100.0 / t.total, 2) AS porcentaje_ambientes_invalidos,

    mon.moneda_vacia,
    ROUND(mon.moneda_vacia * 100.0 / t.total, 2) AS porcentaje_moneda_vacia,

    op.tipo_operacion_vacia,
    ROUND(op.tipo_operacion_vacia * 100.0 / t.total, 2) AS porcentaje_tipo_operacion_vacia

FROM total_registros t,
     precios_invalidos p,
     metros_cuadrados_invalidos m,
     antiguedad_999 a,
     ambientes_invalidos amb,
     moneda_vacia mon,
     tipo_operacion_vacia op;


In [0]:
%sql
SELECT
    COUNT(*) AS cantidad_grupos_duplicados,
    SUM(total) AS total_registros_duplicados,
    SUM(total) - COUNT(*) AS registros_extra_por_duplicacion
FROM (
    SELECT COUNT(*) AS total, precio, url
    FROM bootcamp.bronze.properties_bronze
    GROUP BY precio, url
    HAVING COUNT(*) > 1
) t

In [0]:
%sql
Select precio, url, Count(*) as total
FROM bootcamp.bronze.properties_bronze
Group BY precio, url
Having Count(*) > 1
Order BY total desc
LIMIT 10

In [0]:
%sql
with percentil_1 AS (
    SELECT PERCENTILE(precio, 0.01) as percentil1, moneda
    FROM bootcamp.bronze.properties_bronze
    GROUP BY moneda
), percentil_99 AS (
    SELECT PERCENTILE(precio,0.99) as percentil99, moneda
    FROM bootcamp.bronze.properties_bronze
    GROUP BY moneda
), joins AS(
    Select p1.moneda, p1.percentil1, p99.percentil99
    From percentil_1 p1
    JOIN percentil_99 p99
    ON p1.moneda = p99.moneda
)

SELECT pb.*
FROM bootcamp.bronze.properties_bronze pb
JOIN joins j
  ON pb.moneda = j.moneda
WHERE TRY_CAST(pb.precio AS DOUBLE) < TRY_CAST(j.percentil1 AS DOUBLE)
   OR TRY_CAST(pb.precio AS DOUBLE) > TRY_CAST(j.percentil99 AS DOUBLE)

ORDER BY precio ASC;





## Analisis Avanzado y documentacion

In [0]:
%sql
Select RANK() OVER (ORDER BY AVG(CAST(precio AS DOUBLE)) DESC) as ranking, AVG(CAST(precio AS DOUBLE)) as precio_promedio, zona, Count(*) as total_propiedades
FROM bootcamp.bronze.properties_bronze
Group BY zona

In [0]:
%sql
with promedio_zona AS (
    Select AVG(CAST(precio AS DOUBLE)) as precio_promedio, zona
    FROM bootcamp.bronze.properties_bronze
    Group BY zona
), promedio_total AS (
    SELECT AVG(CAST(precio AS DOUBLE)) as precio_promedio_total
    FROM bootcamp.bronze.properties_bronze
)

Select pz.zona, pz.precio_promedio, pt.precio_promedio_total, pt.precio_promedio_total - pz.precio_promedio as Diferencia
FROM promedio_zona pz 
CROSS JOIN promedio_total pt



In [0]:
%sql


In [0]:
# Resumen ejecutivo de calidad de datos

# 1. Total de registros analizados
total_registros = spark.sql("""
    SELECT COUNT(*) AS total
    FROM bootcamp.bronze.properties_bronze
""").collect()[0]['total']

# 2. Principales problemas de calidad encontrados
problemas_calidad = spark.sql("""
    SELECT 
        precios_invalidos,
        metros_cuadrados_invalidos,
        antiguedad_999,
        ambientes_invalidos,
        moneda_vacia,
        tipo_operacion_vacia
    FROM (
        WITH total_registros AS (
            SELECT COUNT(*) AS total
            FROM bootcamp.bronze.properties_bronze
        ), metros_cuadrados_invalidos AS (
            SELECT COUNT(*) AS metros_cuadrados_invalidos
            FROM bootcamp.bronze.properties_bronze
            WHERE metros_cuadrados_totales IS NULL OR TRY_CAST(metros_cuadrados_totales AS DOUBLE) <= 0
        ), precios_invalidos AS(
            SELECT COUNT(*) AS precios_invalidos
            FROM bootcamp.bronze.properties_bronze
            WHERE precio IS NULL OR TRY_CAST(precio AS DOUBLE) <= 0
        ), moneda_vacia AS (
            SELECT COUNT(*) AS moneda_vacia
            FROM bootcamp.bronze.properties_bronze
            WHERE moneda IS NULL OR moneda = ''
        ), antiguedad_999 AS (
            SELECT COUNT(*) AS antiguedad_999
            FROM bootcamp.bronze.properties_bronze
            WHERE TRY_CAST(antiguedad AS DOUBLE) = 999
        ), ambientes_invalidos AS (
            SELECT COUNT(*) AS ambientes_invalidos
            FROM bootcamp.bronze.properties_bronze
            WHERE ambientes IS NULL OR TRY_CAST(ambientes AS DOUBLE) = 0
        ), tipo_operacion_vacia AS (
            SELECT COUNT(*) AS tipo_operacion_vacia
            FROM bootcamp.bronze.properties_bronze
            WHERE tipo_de_operacion IS NULL OR TRIM(tipo_de_operacion) = ''
        )
        SELECT 
            p.precios_invalidos,
            m.metros_cuadrados_invalidos,
            a.antiguedad_999,
            amb.ambientes_invalidos,
            mon.moneda_vacia,
            op.tipo_operacion_vacia
        FROM precios_invalidos p,
             metros_cuadrados_invalidos m,
             antiguedad_999 a,
             ambientes_invalidos amb,
             moneda_vacia mon,
             tipo_operacion_vacia op
    )
""").collect()[0]

# 3. Porcentaje de datos válidos vs inválidos
porcentajes = spark.sql("""
    WITH total_registros AS (
        SELECT COUNT(*) AS total
        FROM bootcamp.bronze.properties_bronze
    ), metros_cuadrados_invalidos AS (
        SELECT COUNT(*) AS metros_cuadrados_invalidos
        FROM bootcamp.bronze.properties_bronze
        WHERE metros_cuadrados_totales IS NULL OR TRY_CAST(metros_cuadrados_totales AS DOUBLE) <= 0
    ), precios_invalidos AS(
        SELECT COUNT(*) AS precios_invalidos
        FROM bootcamp.bronze.properties_bronze
        WHERE precio IS NULL OR TRY_CAST(precio AS DOUBLE) <= 0
    ), moneda_vacia AS (
        SELECT COUNT(*) AS moneda_vacia
        FROM bootcamp.bronze.properties_bronze
        WHERE moneda IS NULL OR moneda = ''
    ), antiguedad_999 AS (
        SELECT COUNT(*) AS antiguedad_999
        FROM bootcamp.bronze.properties_bronze
        WHERE TRY_CAST(antiguedad AS DOUBLE) = 999
    ), ambientes_invalidos AS (
        SELECT COUNT(*) AS ambientes_invalidos
        FROM bootcamp.bronze.properties_bronze
        WHERE ambientes IS NULL OR TRY_CAST(ambientes AS DOUBLE) = 0
    ), tipo_operacion_vacia AS (
        SELECT COUNT(*) AS tipo_operacion_vacia
        FROM bootcamp.bronze.properties_bronze
        WHERE tipo_de_operacion IS NULL OR TRIM(tipo_de_operacion) = ''
    )
    SELECT 
        t.total,
        p.precios_invalidos,
        m.metros_cuadrados_invalidos,
        a.antiguedad_999,
        amb.ambientes_invalidos,
        mon.moneda_vacia,
        op.tipo_operacion_vacia,
        ROUND(
            100.0 * (
                p.precios_invalidos +
                m.metros_cuadrados_invalidos +
                a.antiguedad_999 +
                amb.ambientes_invalidos +
                mon.moneda_vacia +
                op.tipo_operacion_vacia
            ) / t.total, 2
        ) AS porcentaje_invalidos,
        ROUND(
            100.0 - (
                100.0 * (
                    p.precios_invalidos +
                    m.metros_cuadrados_invalidos +
                    a.antiguedad_999 +
                    amb.ambientes_invalidos +
                    mon.moneda_vacia +
                    op.tipo_operacion_vacia
                ) / t.total
            ), 2
        ) AS porcentaje_validos
    FROM total_registros t,
         precios_invalidos p,
         metros_cuadrados_invalidos m,
         antiguedad_999 a,
         ambientes_invalidos amb,
         moneda_vacia mon,
         tipo_operacion_vacia op
""").collect()[0]

# 4. Recomendaciones para la limpieza en Silver
recomendaciones = [
    "Eliminar o imputar registros con precios inválidos o nulos.",
    "Corregir o eliminar registros con metros cuadrados totales inválidos.",
    "Filtrar registros con antigüedad igual a 999.",
    "Completar o eliminar registros con ambientes nulos o igual a 0.",
    "Completar o eliminar registros con moneda vacía.",
    "Completar o eliminar registros con tipo de operación vacío.",
    "Eliminar duplicados por precio y url.",
    "Validar outliers en precios según percentiles por moneda."
]

# Mostrar resumen estructurado
resumen = {
    "Total de registros analizados": total_registros,
    "Principales problemas de calidad encontrados": problemas_calidad.asDict(),
    "Porcentaje de datos válidos": porcentajes['porcentaje_validos'],
    "Porcentaje de datos inválidos": porcentajes['porcentaje_invalidos'],
    "Recomendaciones para limpieza en Silver": recomendaciones
}

display(resumen)